# Convert MessagePack to CSV with chDB

In-process ClickHouse SQL in Python. Run `./generate.sh` first to create `data/orders.msgpack`.

MsgPack carries no schema, so the read needs an explicit structure. Then write CSV.

In [ ]:
import os
import chdb

os.chdir("data")

In [ ]:
# MsgPack read needs an explicit structure. Write the result straight to CSV.
chdb.query("""
SELECT * FROM file('orders.msgpack', MsgPack,
  'order_date Date, order_id UInt64, country String, product String, revenue Float64, quantity UInt8')
INTO OUTFILE 'orders_chdb.csv' TRUNCATE FORMAT CSVWithNames
""")

In [ ]:
# Confirm the round-trip by reading the CSV back into a DataFrame.
df = chdb.query("""
SELECT country, count() AS orders, round(sum(revenue), 2) AS revenue
FROM file('orders_chdb.csv')
GROUP BY country
ORDER BY revenue DESC
""", "DataFrame")
df